In [1]:
"""
DAY 7 — Validation and sensitivity checks for the 2-D convection project

This script combines tasks for all three project members:
1) Theory/validation: checks the pure-conduction reference solution T = 1-z and Nu = 1.
2) Numerical modelling: compares grid sizes and simulation durations at Ra = 2500.
3) Results/visualization: creates tables and plots for velocity, Nusselt estimates,
   temperature fields, and incompressibility.

Requirements:
    pip install numpy scipy matplotlib

Run this script from the folder containing this file:
    python day7_validation.py

All output is written to a timestamp-free new folder named day7_results.
If that folder already exists, files are given unique suffixes instead of overwritten.

Scientific note:
This is an educational, simplified 2-D model. The fourth-order streamfunction
formulation and its boundary treatment are diagnostics for a project prototype;
the output should not be presented as a fully validated Earth-mantle simulation.
"""

from pathlib import Path
import csv

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags, eye, kron
from scipy.sparse.linalg import factorized


# ============================================================
# SETTINGS
# ============================================================

RA_TEST = 2500.0
GRID_SIZES = [21, 31, 41]
END_TIMES = [0.0125, 0.025]  # short and longer runs, same initial conditions
OUTPUT_BASE = Path("day7_results")

# Use a conservative explicit timestep.
DIFFUSION_SAFETY = 0.20
ADVECTION_CFL = 0.30
MAX_STEPS = 300000


# ============================================================
# OUTPUT FOLDER — avoid overwriting previous results
# ============================================================

def make_output_folder(base):
    folder = base
    suffix = 2
    while folder.exists():
        folder = Path(f"{base}_{suffix}")
        suffix += 1
    folder.mkdir(parents=True)
    return folder


# ============================================================
# NUMERICAL OPERATORS
# ============================================================

def laplacian_interior(a, dx, dz):
    """Return the 2-D Laplacian at interior grid points."""
    return (
        (a[1:-1, 2:] - 2.0*a[1:-1, 1:-1] + a[1:-1, :-2]) / dx**2
        + (a[2:, 1:-1] - 2.0*a[1:-1, 1:-1] + a[:-2, 1:-1]) / dz**2
    )


def make_biharmonic_solver(n, dx, dz):
    """
    Construct a sparse discrete biharmonic operator and factorize it once.

    This uses a simple zero-boundary streamfunction treatment for the
    educational prototype. Boundary-condition sensitivity should be considered
    before interpreting the model as a quantitatively validated Stokes solver.
    """
    m = n - 2

    Dxx = diags(
        [np.ones(m-1)/dx**2, -2*np.ones(m)/dx**2, np.ones(m-1)/dx**2],
        [-1, 0, 1], format="csc"
    )
    Dzz = diags(
        [np.ones(m-1)/dz**2, -2*np.ones(m)/dz**2, np.ones(m-1)/dz**2],
        [-1, 0, 1], format="csc"
    )

    I = eye(m, format="csc")
    lap = kron(I, Dxx, format="csc") + kron(Dzz, I, format="csc")
    biharmonic = (lap @ lap).tocsc()
    return factorized(biharmonic)


def solve_streamfunction(T, ra, dx, dz, solve_biharmonic):
    """Solve the prototype equation ∇⁴ψ = -Ra * ∂T/∂x."""
    n = T.shape[0]
    dTdx = np.gradient(T, dx, axis=1)
    rhs = -ra * dTdx[1:-1, 1:-1]

    psi = np.zeros_like(T)
    psi[1:-1, 1:-1] = solve_biharmonic(rhs.ravel()).reshape(n-2, n-2)
    return psi


def velocity_from_psi(psi, dx, dz):
    """Use u = ∂ψ/∂z and w = -∂ψ/∂x."""
    u = np.gradient(psi, dz, axis=0)
    w = -np.gradient(psi, dx, axis=1)
    return u, w


def apply_temperature_boundaries(T):
    """Hot bottom, cold top, and approximately insulating sidewalls."""
    T[0, :] = 1.0
    T[-1, :] = 0.0
    T[1:-1, 0] = T[1:-1, 1]
    T[1:-1, -1] = T[1:-1, -2]
    # Reassert fixed top/bottom after sidewall assignments.
    T[0, :] = 1.0
    T[-1, :] = 0.0
    return T


# ============================================================
# ONE CONVECTION RUN
# ============================================================

def run_convection(n, ra, end_time):
    x = np.linspace(0.0, 1.0, n)
    z = np.linspace(0.0, 1.0, n)
    dx = x[1] - x[0]
    dz = z[1] - z[0]
    X, Z = np.meshgrid(x, z)

    # Same initial condition for every test: conductive state + small disturbance.
    T = 1.0 - Z + 0.015*np.sin(np.pi*X)*np.sin(np.pi*Z)
    T = apply_temperature_boundaries(T)

    solve_biharmonic = make_biharmonic_solver(n, dx, dz)
    dt_diff = DIFFUSION_SAFETY / (
        2.0 * (1.0/dx**2 + 1.0/dz**2)
    )

    t = 0.0
    steps = 0

    while t < end_time:
        psi = solve_streamfunction(T, ra, dx, dz, solve_biharmonic)
        u, w = velocity_from_psi(psi, dx, dz)

        speed_max = max(
            float(np.max(np.abs(u))),
            float(np.max(np.abs(w))),
            1e-12
        )
        dt_adv = ADVECTION_CFL * min(dx, dz) / speed_max
        dt = min(dt_diff, dt_adv, end_time - t)

        # First-order upwind advection at interior points.
        uc = u[1:-1, 1:-1]
        wc = w[1:-1, 1:-1]
        Tc = T[1:-1, 1:-1]

        dTdx_upwind = np.where(
            uc >= 0,
            (Tc - T[1:-1, :-2])/dx,
            (T[1:-1, 2:] - Tc)/dx
        )
        dTdz_upwind = np.where(
            wc >= 0,
            (Tc - T[:-2, 1:-1])/dz,
            (T[2:, 1:-1] - Tc)/dz
        )

        T_new = T.copy()
        T_new[1:-1, 1:-1] = Tc + dt * (
            -uc*dTdx_upwind
            -wc*dTdz_upwind
            +laplacian_interior(T, dx, dz)
        )
        T = apply_temperature_boundaries(T_new)

        t += dt
        steps += 1
        if steps > MAX_STEPS:
            raise RuntimeError(
                f"Stopped: maximum steps reached for n={n}, Ra={ra}, t_end={end_time}."
            )

    # Final velocity field from the final temperature.
    psi = solve_streamfunction(T, ra, dx, dz, solve_biharmonic)
    u, w = velocity_from_psi(psi, dx, dz)
    speed = np.hypot(u, w)

    # Boundary heat-flux / Nusselt diagnostics; omit sidewall corner points.
    nu_bottom = -float(np.mean((T[1, 1:-1] - T[0, 1:-1])/dz))
    nu_top = -float(np.mean((T[-1, 1:-1] - T[-2, 1:-1])/dz))
    nu_mismatch = abs(nu_bottom - nu_top) / max(
        0.5*(abs(nu_bottom) + abs(nu_top)), 1e-12
    ) * 100.0

    # Numerical checks.
    finite_ok = bool(
        np.all(np.isfinite(T)) and np.all(np.isfinite(u))
        and np.all(np.isfinite(w)) and np.all(np.isfinite(psi))
    )
    temperature_min = float(np.min(T))
    temperature_max = float(np.max(T))

    # For u=∂ψ/∂z and w=-∂ψ/∂x, divergence should be near zero.
    dudx = np.gradient(u, dx, axis=1)
    dwdz = np.gradient(w, dz, axis=0)
    divergence = dudx + dwdz
    div_rms = float(np.sqrt(np.mean(divergence[1:-1, 1:-1]**2)))
    velocity_scale = max(float(np.sqrt(np.mean(
        u[1:-1, 1:-1]**2 + w[1:-1, 1:-1]**2
    ))), 1e-12)
    div_relative = div_rms / velocity_scale

    return {
        "n": n, "ra": ra, "end_time_requested": end_time,
        "time_reached": t, "steps": steps,
        "x": x, "z": z, "X": X, "Z": Z,
        "T": T, "psi": psi, "u": u, "w": w, "speed": speed,
        "Nu_bottom": nu_bottom, "Nu_top": nu_top,
        "Nu_mismatch_percent": nu_mismatch,
        "max_speed": float(np.max(speed)),
        "max_abs_u": float(np.max(np.abs(u))),
        "max_abs_w": float(np.max(np.abs(w))),
        "T_min": temperature_min, "T_max": temperature_max,
        "finite_ok": finite_ok,
        "divergence_rms": div_rms,
        "relative_divergence_rms": div_relative,
    }


# ============================================================
# PURE CONDUCTION VALIDATION (NO FLOW)
# ============================================================

def run_conduction_validation(n=41):
    """
    Solve transient conduction with zero velocity and compare with T=1-z.
    The test is independent of the convection solver.
    """
    x = np.linspace(0.0, 1.0, n)
    z = np.linspace(0.0, 1.0, n)
    dx = x[1] - x[0]
    dz = z[1] - z[0]
    X, Z = np.meshgrid(x, z)

    T = 1.0 - Z + 0.02*np.sin(np.pi*X)*np.sin(np.pi*Z)
    T = apply_temperature_boundaries(T)

    dt = 0.20 / (2.0*(1.0/dx**2 + 1.0/dz**2))
    tolerance = 1e-8
    max_steps = 300000
    steps = 0
    change = np.inf

    while change > tolerance and steps < max_steps:
        T_old = T.copy()
        T_new = T.copy()
        T_new[1:-1, 1:-1] = T_old[1:-1, 1:-1] + dt*laplacian_interior(
            T_old, dx, dz
        )
        T = apply_temperature_boundaries(T_new)
        change = float(np.max(np.abs(T - T_old)))
        steps += 1

    analytical = 1.0 - Z
    error = np.abs(T - analytical)
    return {
        "n": n, "steps": steps, "dt": dt,
        "max_error": float(np.max(error)),
        "rms_error": float(np.sqrt(np.mean(error**2))),
        "final_change": change,
        "T": T, "analytical": analytical, "X": X, "Z": Z,
        "converged": change <= tolerance,
    }


# ============================================================
# MAIN: ALL THREE MEMBERS' TASKS
# ============================================================

def main():
    output_dir = make_output_folder(OUTPUT_BASE)
    print(f"Results folder: {output_dir.resolve()}")

    # ---- Person 1: validate the pure-conduction reference ----
    print("\nPERSON 1 — PURE CONDUCTION VALIDATION")
    cond = run_conduction_validation(n=41)
    print(f"Grid: {cond['n']} x {cond['n']}")
    print(f"Iterations: {cond['steps']}")
    print(f"Convergence reached: {cond['converged']}")
    print(f"Max error against T=1-z: {cond['max_error']:.6e}")
    print(f"RMS error against T=1-z: {cond['rms_error']:.6e}")
    print("Expected conductive reference: Nu = 1")

    fig, ax = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    im = ax[0].contourf(cond["X"], cond["Z"], cond["T"], levels=21, cmap="inferno")
    ax[0].set_title("Numerical conduction")
    ax[1].contourf(cond["X"], cond["Z"], cond["analytical"], levels=21, cmap="inferno")
    ax[1].set_title("Analytical T = 1-z")
    error = np.abs(cond["T"] - cond["analytical"])
    im_err = ax[2].contourf(cond["X"], cond["Z"], error, levels=21, cmap="viridis")
    ax[2].set_title("Absolute error")
    for a in ax:
        a.set_xlabel("x / d")
        a.set_ylabel("z / d")
        a.set_aspect("equal")
    fig.colorbar(im, ax=ax[:2], shrink=0.8, label="Dimensionless temperature")
    fig.colorbar(im_err, ax=ax[2], shrink=0.8, label="Absolute error")
    fig.savefig(output_dir / "day7_conduction_validation.png", dpi=180)
    plt.close(fig)

    # ---- Person 2: grid and time sensitivity at Ra=2500 ----
    print("\nPERSON 2 — GRID AND TIME SENSITIVITY")
    runs = []
    for n in GRID_SIZES:
        for end_time in END_TIMES:
            print(f"Running Ra={RA_TEST:g}, grid={n}x{n}, end_time={end_time:g} ...")
            r = run_convection(n, RA_TEST, end_time)
            runs.append(r)
            print(
                f"  steps={r['steps']}, max_speed={r['max_speed']:.5g}, "
                f"Nu_bottom={r['Nu_bottom']:.5f}, Nu_top={r['Nu_top']:.5f}"
            )

    # Save summary CSV.
    csv_path = output_dir / "day7_convergence_summary.csv"
    columns = [
        "n", "ra", "end_time_requested", "time_reached", "steps",
        "max_speed", "max_abs_u", "max_abs_w",
        "Nu_bottom", "Nu_top", "Nu_mismatch_percent",
        "T_min", "T_max", "finite_ok",
        "divergence_rms", "relative_divergence_rms"
    ]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for r in runs:
            writer.writerow({key: r[key] for key in columns})

    # ---- Person 3: plots and interpretation support ----
    # Plot maximum speed by grid size, with separate curves for final time.
    fig, ax = plt.subplots(figsize=(7, 5))
    for end_time in END_TIMES:
        selected = [r for r in runs if r["end_time_requested"] == end_time]
        selected.sort(key=lambda r: r["n"])
        ax.plot(
            [r["n"] for r in selected],
            [r["max_speed"] for r in selected],
            "o-", label=f"t = {end_time:g}"
        )
    ax.set_xlabel("Grid points per dimension")
    ax.set_ylabel("Maximum dimensionless speed")
    ax.set_title(f"Grid sensitivity at Ra={RA_TEST:g}")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "day7_grid_sensitivity_speed.png", dpi=180)
    plt.close(fig)

    # Plot Nusselt estimates for all grid/time combinations.
    labels = [f"{r['n']}², t={r['end_time_requested']:g}" for r in runs]
    positions = np.arange(len(runs))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(positions, [r["Nu_bottom"] for r in runs], "o-", label="Bottom Nu")
    ax.plot(positions, [r["Nu_top"] for r in runs], "s--", label="Top Nu")
    ax.axhline(1.0, color="black", linestyle=":", label="Conduction reference Nu=1")
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylabel("Nusselt estimate")
    ax.set_title(f"Boundary heat-flux diagnostics at Ra={RA_TEST:g}")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "day7_nusselt_sensitivity.png", dpi=180)
    plt.close(fig)

    # Plot temperature and velocity for the finest grid at the longer time.
    representative = [
        r for r in runs
        if r["n"] == max(GRID_SIZES) and r["end_time_requested"] == max(END_TIMES)
    ][0]
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
    im = ax[0].contourf(
        representative["X"], representative["Z"], representative["T"],
        levels=np.linspace(0, 1, 21), cmap="inferno", extend="both"
    )
    ax[0].contour(
        representative["X"], representative["Z"], representative["T"],
        levels=np.linspace(0.1, 0.9, 9), colors="white", linewidths=0.4
    )
    ax[0].set_title("Temperature field")
    ax[1].streamplot(
        representative["x"], representative["z"],
        representative["u"], representative["w"],
        color=representative["speed"], cmap="viridis", density=1.0
    )
    ax[1].set_title("Streamlines")
    for a in ax:
        a.set_xlabel("x / d")
        a.set_ylabel("z / d")
        a.set_aspect("equal")
    fig.colorbar(im, ax=ax[0], label="Dimensionless temperature", shrink=0.8)
    fig.savefig(output_dir / "day7_representative_fields.png", dpi=180)
    plt.close(fig)

    # Save the representative data for later analysis.
    np.savez_compressed(
        output_dir / "day7_representative_Ra2500.npz",
        x=representative["x"], z=representative["z"],
        T=representative["T"], psi=representative["psi"],
        u=representative["u"], w=representative["w"],
        speed=representative["speed"],
        ra=representative["ra"], time=representative["time_reached"],
        steps=representative["steps"]
    )

    print("\nDAY 7 SUMMARY")
    print(f"CSV table: {csv_path.resolve()}")
    print(f"All plots and data saved in: {output_dir.resolve()}")
    print("\nHow to interpret:")
    print("- Similar values across grids suggest reduced grid sensitivity, not proof of full correctness.")
    print("- Similar results at the two end times suggest reduced time sensitivity.")
    print("- Large top/bottom Nu mismatch can indicate a transient state or numerical/boundary error.")
    print("- Temperature outside [0,1] or non-finite values should be investigated.")
    print("- A low divergence residual is expected from a streamfunction-derived velocity, but boundary/discretization effects remain.")
    print("- The conductive test should approach T=1-z and have Nu=1.")
    print("\nThis prototype needs further boundary-condition and solver validation before quantitative geophysical claims.")


if __name__ == "__main__":
    main()

Results folder: C:\Users\athil\OneDrive\Desktop\day7_results

PERSON 1 — PURE CONDUCTION VALIDATION
Grid: 41 x 41
Iterations: 19454
Convergence reached: True
Max error against T=1-z: 3.241966e-05
RMS error against T=1-z: 2.264287e-05
Expected conductive reference: Nu = 1

PERSON 2 — GRID AND TIME SENSITIVITY
Running Ra=2500, grid=21x21, end_time=0.0125 ...
  steps=100, max_speed=0.073719, Nu_bottom=0.97248, Nu_top=1.02741
Running Ra=2500, grid=21x21, end_time=0.025 ...
  steps=200, max_speed=0.023018, Nu_bottom=0.97577, Nu_top=1.02415
Running Ra=2500, grid=31x31, end_time=0.0125 ...
  steps=226, max_speed=0.077947, Nu_bottom=0.97278, Nu_top=1.02711
Running Ra=2500, grid=31x31, end_time=0.025 ...
  steps=451, max_speed=0.024376, Nu_bottom=0.97600, Nu_top=1.02391
Running Ra=2500, grid=41x41, end_time=0.0125 ...
  steps=400, max_speed=0.079992, Nu_bottom=0.97293, Nu_top=1.02695
Running Ra=2500, grid=41x41, end_time=0.025 ...
  steps=800, max_speed=0.025056, Nu_bottom=0.97613, Nu_top=1.023